# Bước 15 – Lần chạy chốt: 5 hạt giống ngẫu nhiên, khoảng tin cậy, kiểm định thống kê

Đây là lần chạy cho SỐ LIỆU CUỐI CÙNG của bài báo. Cấu hình đã chốt (không chỉnh thêm):
- 3 loại mô hình: LightGBM, MLP, cây quyết định;
- hướng 1 (không nhãn): trung bình đều, chọn theo kiểm định nguồn, TADP, CORAL, Isolation Forest, tự huấn luyện, nhãn giả đồng thuận;
- hướng 2: M2 với 100 / 500 / 1.000 / 5.000 nhãn thật, bốc 3 lần;
- mốc đủ nhãn;
- ngưỡng: cố định (a), Otsu có chặn (e), chống sập 5 / 10 / 20% (h10 là cách chính).

Lý thuyết (khoảng tin cậy, kiểm định Wilcoxon, kèm ví dụ số) ở Phần 7 của tài liệu "TADP-IDS – Lý thuyết bước 13".

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Chạy 5 hạt giống × 3 loại mô hình × 4 mạng | khoảng 4–6 giờ tổng cộng |
| 3 | Tổng hợp + kiểm định (chạy được bất cứ lúc nào) | 1 phút |

**Dùng máy CPU, không GPU** (Runtime → Change runtime type → CPU). Kết quả lưu vào `MyDrive/TADP_Q1/results/b15_chot/`.

**Nếu Colab ngắt giữa chừng:** mở lại notebook, chạy lại Ô 1 rồi Ô 2 – phần nào xong rồi (hạt giống, loại mô hình, mạng) sẽ được bỏ qua, chạy tiếp từ chỗ dừng. Có thể chạy làm nhiều buổi.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [2]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
import torch
print("GPU:", "CÓ – nên đổi sang CPU" if torch.cuda.is_available() else "không có (đúng như mong muốn)")
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "experts.py": 5934,
            "few_label.py": 15932, "final_run.py": 18041, "label_free.py": 16327, "metrics.py": 4002,
            "model_families.py": 10460, "preprocess.py": 3601, "shift.py": 3863, "window_threshold.py": 12240}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 15" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
GPU: không có (đúng như mong muốn)
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 15


## Ô 2 – Chạy (dài)

In lần lượt "[hạt giống …] [loại mô hình] 4 chuyên gia nguồn xong", rồi "[hạt giống …] [loại mô hình] [tên mạng] xong trong … giây". Tổng cộng 5 × 3 × 4 = 60 dòng "xong". Chạy xong toàn bộ thì in luôn các bảng tổng hợp.

In [3]:
import sys, warnings
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.final_run import run_final
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b15_chot",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode", LGBM_ROUNDS=200)
data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
run_final(cfg, data, splits, cols, pv="P1_log")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
[hạt giống 11] [lgbm] 4 chuyên gia nguồn xong (93 giây)
[hạt giống 11] [lgbm] [CIC-IDS2017] xong trong 375 giây
[hạt giống 11] [lgbm] [CSE-CIC-IDS2018] xong trong 386 giây
[hạt giống 11] [lgbm] [CIC-ToN-IoT] xong trong 377 giây
[hạt giống 11] [lgbm] [CIC-UNSW-NB15] xong trong 370 giây
[hạt giống 11] [mlp] 4 chuyên gia nguồn xong (103 giây)
[hạt giống 11] [mlp] [CIC-IDS2017] xong trong 293 giây
[hạt giống 11] [mlp] [CSE-CIC-IDS2018] xong trong 336 giây
[hạt giống 11] [mlp] [CIC-ToN-IoT] xong trong 338 giây
[hạt giống 11] [mlp] [CIC-UNSW-NB15] xong trong 448 giây
[hạt giống 11] [tree] 4 chuyên gia nguồn xong (50 giây)
[

,mo_hinh,phuong_phap,k,cach_nguong,macro_f1,macro_f1_ci,auroc,auroc_ci,o,oa
0,lgbm,CORAL,0,a_co_dinh,0.442615,0.032912,0.529081,0.082000,0.443 ± 0.033,0.529 ± 0.082
1,lgbm,CORAL,0,e_otsu_cua_so_co_chan,0.350663,0.041545,0.529081,0.082000,0.351 ± 0.042,0.529 ± 0.082
2,lgbm,CORAL,0,h_chong_sap_05,0.350663,0.041545,0.529081,0.082000,0.351 ± 0.042,0.529 ± 0.082
3,lgbm,CORAL,0,h_chong_sap_10,0.350663,0.041545,0.529081,0.082000,0.351 ± 0.042,0.529 ± 0.082
4,lgbm,CORAL,0,h_chong_sap_20,0.350663,0.041545,0.529081,0.082000,0.351 ± 0.042,0.529 ± 0.082
5,lgbm,DE_XUAT_dong_thuan,0,a_co_dinh,0.469573,0.025202,0.588361,0.044033,0.470 ± 0.025,0.588 ± 0.044
6,lgbm,DE_XUAT_dong_thuan,0,e_otsu_cua_so_co_chan,0.471180,0.022580,0.588361,0.044033,0.471 ± 0.023,0.588 ± 0.044
7,lgbm,DE_XUAT_dong_thuan,0,h_chong_sap_05,0.471180,0.022580,0.588361,0.044033,0.471 ± 0.023,0.588 ± 0.044
8,lgbm,DE_XUAT_dong_thuan,0,h_chong_sap_10,0.471180,0.022580,0.588361,0.044033,0.471 ± 0.023,0.588 ± 0.044
9,lgbm,DE_XUAT_dong_thuan,0,h_chong_sap_20,0.471180,0.022580,0.588361,0.044033,0.471 ± 0.023,0.588 ± 0.044


## Ô 3 – Tổng hợp và kiểm định (chạy bất cứ lúc nào, kể cả khi mới xong một phần)

- Bảng 15.1: Macro-F1 ± nửa khoảng tin cậy 95%; Bảng 15.2: AUROC ± nửa khoảng tin cậy 95%;
- Bảng 15.3: kiểm định Wilcoxon – ngưỡng h10 so với ngưỡng cố định;
- Bảng 15.4: kiểm định Wilcoxon – M2 với k nhãn so với cách không nhãn tốt nhất;
- Bảng 15.5: từng mạng; Bảng 15.6: độ nhạy mức chống sập; Bảng 15.7: số lần M2 không học được.

Chạy xong báo Claude.

In [4]:
import sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.final_run import summarize_final
B = "/content/drive/MyDrive/TADP_Q1"
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b15_chot")
summarize_final(cfg)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Số hạt giống đã có: 5 ([np.int64(11), np.int64(22), np.int64(33), np.int64(44), np.int64(55)])

BẢNG 15.1 – Macro-F1 trung bình 4 mạng ± nửa khoảng tin cậy 95% (qua các hạt giống)
  a = ngưỡng cố định, h10 = chống sập 10% (cách chính); k = -1 là mốc đủ nhãn
 mo_hinh                                 lgbm                           mlp                          tree               
cach_nguong                        a_co_dinh h_chong_sap_10      a_co_dinh h_chong_sap_10      a_co_dinh h_chong_sap_10
phuong_phap             k                                                                                              
CORAL                    0     0.443 ± 0.033  0.351 ± 0.042  0.413 ± 0.073  0.444 ± 0.055  0.440 ± 0.013  0.330 ± 0.039
DE_XUAT_dong_thuan       0     0.470 ± 0.025  0.471 ± 0.023  0.463 ± 0.066  0.476 ± 0.073  0.451 ± 0.027  0.466 ± 0.041
M2          

,mo_hinh,phuong_phap,k,cach_nguong,macro_f1,macro_f1_ci,auroc,auroc_ci,o,oa
0,lgbm,CORAL,0,a_co_dinh,0.442615,0.032912,0.529081,0.082000,0.443 ± 0.033,0.529 ± 0.082
1,lgbm,CORAL,0,e_otsu_cua_so_co_chan,0.350663,0.041545,0.529081,0.082000,0.351 ± 0.042,0.529 ± 0.082
2,lgbm,CORAL,0,h_chong_sap_05,0.350663,0.041545,0.529081,0.082000,0.351 ± 0.042,0.529 ± 0.082
3,lgbm,CORAL,0,h_chong_sap_10,0.350663,0.041545,0.529081,0.082000,0.351 ± 0.042,0.529 ± 0.082
4,lgbm,CORAL,0,h_chong_sap_20,0.350663,0.041545,0.529081,0.082000,0.351 ± 0.042,0.529 ± 0.082
5,lgbm,DE_XUAT_dong_thuan,0,a_co_dinh,0.469573,0.025202,0.588361,0.044033,0.470 ± 0.025,0.588 ± 0.044
6,lgbm,DE_XUAT_dong_thuan,0,e_otsu_cua_so_co_chan,0.471180,0.022580,0.588361,0.044033,0.471 ± 0.023,0.588 ± 0.044
7,lgbm,DE_XUAT_dong_thuan,0,h_chong_sap_05,0.471180,0.022580,0.588361,0.044033,0.471 ± 0.023,0.588 ± 0.044
8,lgbm,DE_XUAT_dong_thuan,0,h_chong_sap_10,0.471180,0.022580,0.588361,0.044033,0.471 ± 0.023,0.588 ± 0.044
9,lgbm,DE_XUAT_dong_thuan,0,h_chong_sap_20,0.471180,0.022580,0.588361,0.044033,0.471 ± 0.023,0.588 ± 0.044
